In [36]:
!pip install torch
!pip install scipy
!pip install numpy


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.io.wavfile as wav

In [38]:
def getWav(path):
    rate, data = wav.read(path)

    wave = torch.tensor(data, dtype=torch.float32) / 32768.0

    if wave.ndim > 1:
        wave = wave.mean(dim=1)

    wave = wave.unsqueeze(0).unsqueeze(0)

    return wave, rate

In [39]:
wav, rate = getWav("sound.wav")

In [40]:
wav

tensor([[[0.0125, 0.0115, 0.0111,  ..., 0.0000, 0.0000, 0.0000]]])

In [48]:
def split_wav_into_chunks(wave, chunk_size=2048, overlap=0):
    """
    Splits waveform into fixed-size chunks.

    wave: [1, 1, T] or [T]
    returns: list of [chunk_size] tensors
    """

    # ---- reshape ----
    if wave.ndim == 3:
        x = wave.squeeze(0).squeeze(0)
    elif wave.ndim == 2:
        x = wave.squeeze(0)
    else:
        x = wave

    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(x) - chunk_size, step):
        chunk = x[i:i + chunk_size]
        chunks.append(chunk)

    return chunks

In [49]:
chunks = split_wav_into_chunks(wav)

print(len(chunks))
print(chunks[0].shape)

119
torch.Size([2048])


In [50]:
import wave
import torch
import numpy

def save_chunks(chunks, rate, folder="chunks_out"):
    import os
    os.makedirs(folder, exist_ok=True)

    for i, c in enumerate(chunks):
        # ensure 1D tensor
        audio = c.detach().cpu().flatten()

        # convert to int16
        audio = (audio * 32768.0).clamp(-32768, 32767).to(torch.int16)

        path = os.path.join(folder, f"chunk_{i}.wav")

        with wave.open(path, "wb") as f:
            f.setnchannels(1)
            f.setsampwidth(2)  # int16 = 2 bytes
            f.setframerate(rate)
            f.writeframes(audio.numpy().tobytes())

    print(f"Saved {len(chunks)} chunks to {folder}")

In [51]:
save_chunks(chunks, rate)

RuntimeError: Numpy is not available